# SeineCrops — Sprint S3 : Classification des cultures

Classification supervisée des parcelles RPG à partir des séries
temporelles Sentinel-2 agrégées en S2. Le modèle reçoit en entrée
les **704 features** de `derived.s2_parcelles_monthly` (11 variables
× 4 statistiques × 16 mois) pivotées en format wide (une ligne par
parcelle). Les codes cultures RPG sont regroupés en **7–8 classes**
(blé tendre, orge, colza, maïs, betterave, lin, prairies, autres).

---

## Approche

**Baseline** : Random Forest (scikit-learn), choisi pour sa robustesse
aux features non normalisées et sa lisibilité (feature importances).
Le split est **spatial par blocs** — pas de split aléatoire, qui
créerait une fuite spatiale entre parcelles voisines partageant le
même contexte pédo-climatique et des profils temporels corrélés.

**Évaluation** : matrice de confusion, F1 macro et F1 par classe,
avec attention portée aux classes minoritaires (lin, betterave).
Cible indicative : **F1 macro ≥ 0,85** sur les grandes cultures.

**Option DL** : 1D-CNN ou LSTM sur la dimension temporelle (16 pas
de temps), à envisager si la baseline Random Forest ne satisfait pas
les critères de validation.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| 4.1 | Préparation du feature set — pivot wide, regroupement des cultures, diagnostic des valeurs manquantes |
| 4.2 | Split spatial par blocs — découpage géographique train / test sans fuite spatiale |
| 4.3 | Entraînement Random Forest — baseline, tuning hyperparamètres |
| 4.4 | Évaluation — matrice de confusion, F1 macro / par classe, feature importances |

---

## Références

- [scikit-learn — RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [Spatial cross-validation — blocage géographique (Roberts et al., 2017)](https://doi.org/10.1111/ecog.02881)
- `cadrage/methode.md` — §S3 Classification
- `03_series_s2.ipynb` — sprint S2, `derived.s2_parcelles_monthly` (feature set source)
- `01_ingestion_rpg.ipynb` — sprint S1, `derived.rpg_parcelles_aoi` (cultures déclarées)

### 4.1 — Préparation du feature set

Construction de la matrice d'entrée du modèle à partir de
`derived.s2_parcelles_monthly` (format long : une ligne par
parcelle × mois × variable) et de `derived.rpg_parcelles_aoi`
(culture déclarée).

**Pivot wide** : la table longue est pivotée en une matrice
(parcelles × features) de **704 colonnes** — une par combinaison
`{variable}_{stat}_{mois}` (ex. `ndvi_mean_2024-06`). Chaque ligne
correspond à une parcelle identifiée par `id_parcel`.

**Regroupement des cultures** : les codes `code_group` du RPG sont
regroupés en 7–8 classes cibles adaptées à l'openfield normand :
blé tendre, orge, colza, maïs, betterave, lin, prairies, autres.
Le regroupement est défini dans un dictionnaire explicite pour
assurer la traçabilité et permettre un ajustement ultérieur des
seuils de classe.

**Diagnostic NaN** : les 2 751 parcelles sans pixel capturé à 20 m
(identifiées en S2.4) sont absentes de la table longue et donc du
pivot. Pour les parcelles présentes, les valeurs manquantes
(mois × variable sans composite valide) sont diagnostiquées :
taux de NaN global et par variable/mois, distribution par classe.
La stratégie d'imputation (ou d'exclusion) est décidée après
diagnostic.

In [ ]:
# ── Imports (communs à toutes les sections) ───────────────────────────
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# ── Racine du projet ──────────────────────────────────────────────────
def find_project_root(marker: str = ".projectroot") -> Path:
    here = Path().resolve()
    for parent in [here, *here.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Racine du projet introuvable")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

# ── Connexion PostGIS ─────────────────────────────────────────────────
DB_PARAMS = {
    "host": os.getenv("PG_HOST", "localhost"),
    "port": os.getenv("PG_PORT", 5432),
    "dbname": os.getenv("PG_DBNAME", "seinecrops"),
    "user": os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD"),
}

In [ ]:
# --- Chargement depuis PostGIS ---------------------------------------

conn = psycopg2.connect(**DB_PARAMS)

# Feature set long : ~14 M lignes
sql_features = """
    SELECT id_parcel, mois, variable, mean, std, p10, p90
    FROM derived.s2_parcelles_monthly
    ORDER BY id_parcel, mois, variable
"""
df_long = pd.read_sql(sql_features, conn)

# Labels RPG : code_group par parcelle
sql_labels = """
    SELECT id_parcel, code_group
    FROM derived.rpg_parcelles_aoi
"""
df_labels = pd.read_sql(sql_labels, conn)

conn.close()

print(f"Features long : {len(df_long):>12,} lignes")
print(f"Parcelles dans features : {df_long['id_parcel'].nunique():>7,}")
print(f"Parcelles RPG : {len(df_labels):>12,} parcelles")

In [ ]:
# --- Pivot long → wide -----------------------------------------------

STATS = ["mean", "std", "p10", "p90"]

# Melt stats en une seule colonne (stat, value) pour pivot propre
df_melt = df_long.melt(
    id_vars=["id_parcel", "mois", "variable"],
    value_vars=STATS,
    var_name="stat",
    value_name="value",
)

# Clé de colonne : variable_stat_mois (ex. ndvi_mean_2024-06)
df_melt["feature"] = (
    df_melt["variable"] + "_" + df_melt["stat"] + "_" + df_melt["mois"]
)

# Pivot : une ligne par parcelle, une colonne par feature
df_wide = df_melt.pivot(
    index="id_parcel", columns="feature", values="value"
)
df_wide.columns.name = None

print(f"Matrice wide : {df_wide.shape[0]:,} parcelles × {df_wide.shape[1]} features")
assert df_wide.shape[1] == 704, f"Attendu 704 features, obtenu {df_wide.shape[1]}"

In [ ]:
# --- Regroupement des cultures (v3 : 8 classes) ----------------------

conn = psycopg2.connect(**DB_PARAMS)

sql_labels = """
    SELECT id_parcel, code_group::int AS code_group, code_cultu
    FROM derived.rpg_parcelles_aoi
"""
df_labels = pd.read_sql(sql_labels, conn)
conn.close()

# Dédupliquer les 6 id_parcel en doublon dans le RPG
df_labels = df_labels.drop_duplicates(subset="id_parcel", keep="first")

# Mapping code_group → classe cible (v3 : 8 classes)
GROUP_MAP = {
    1:  "cereales_hiver",      # Blé tendre
    3:  "cereales_hiver",      # Orge
    2:  "mais",                # Maïs grain et ensilage
    5:  "colza",               # Colza
    9:  "lin",                 # Plantes à fibres (≈ lin fibre en Normandie)
    18: "prairie",             # Prairies permanentes
    19: "prairie",             # Prairies temporaires
    25: "legumes_fleurs",      # Légumes ou fleurs
}

# Mapping en deux temps : d'abord code_group, puis exception betterave
df_labels["classe"] = df_labels["code_group"].map(GROUP_MAP).fillna("autres")
df_labels.loc[df_labels["code_cultu"] == "BTN", "classe"] = "betterave"

# Supprimer la colonne classe si déjà présente (ré-exécution)
df_wide = df_wide.drop(columns=["classe"], errors="ignore")

# Jointure avec le feature set
df_wide = df_wide.join(
    df_labels.set_index("id_parcel")

In [ ]:
# --- Diagnostic valeurs manquantes -----------------------------------

n_total = df_wide.shape[0] * 704
n_nan = df_wide.drop(columns=["classe"]).isna().sum().sum()

print(f"NaN : {n_nan:,} / {n_total:,} ({100 * n_nan / n_total:.2f} %)\n")

# Taux de NaN par mois
feature_cols = [c for c in df_wide.columns if c != "classe"]
nan_by_month = {}
for col in feature_cols:
    mois = col.rsplit("_", 1)[-1]       # ndvi_mean_2024-06 → 2024-06
    nan_by_month.setdefault(mois, []).append(df_wide[col].isna().sum())

print("NaN par mois (total sur toutes features) :")
for mois in sorted(nan_by_month):
    total = sum(nan_by_month[mois])
    pct = 100 * total / (len(nan_by_month[mois]) * len(df_wide))
    print(f"  {mois} : {total:>8,}  ({pct:.2f} %)")

### 4.2 — Split spatial par blocs

Découpage train / test réduisant la fuite spatiale. Un split
aléatoire placerait des parcelles voisines — partageant le même
contexte pédo-climatique et des profils temporels corrélés — de
part et d'autre de la frontière train/test, gonflant
artificiellement les métriques d'évaluation.

**Méthode** : l'emprise de l'AOI est découpée en blocs
géographiques réguliers (grille carrée). Chaque bloc est affecté
intégralement au train ou au test, de sorte que les parcelles
d'un même voisinage local restent ensemble. Aux frontières entre
blocs, des parcelles train et test se jouxtent — l'autocorrélation
est réduite, pas éliminée. La taille de bloc est un compromis :
trop petite, l'effet de bloc s'efface ; trop grande, le nombre
de blocs diminue et la variance du split augmente.

**Ratio cible** : ~80 % train / 20 % test en nombre de parcelles,
avec vérification que chaque classe conserve une représentation
suffisante dans les deux ensembles.

**Géométrie** : les centroïdes des parcelles (issus de
`derived.rpg_parcelles_aoi`) servent à affecter chaque parcelle
à un bloc. La grille est alignée sur l'emprise de l'AOI en
EPSG:2154.

In [ ]:
# --- Centroïdes des parcelles ----------------------------------------

conn = psycopg2.connect(**DB_PARAMS)

sql_centroids = """
    SELECT id_parcel,
           ST_X(ST_Centroid(geom)) AS cx,
           ST_Y(ST_Centroid(geom)) AS cy
    FROM derived.rpg_parcelles_aoi
"""
df_centr = pd.read_sql(sql_centroids, conn)
conn.close()

df_centr = df_centr.drop_duplicates(subset="id_parcel", keep="first")

# Ne garder que les parcelles présentes dans le feature set
df_centr = df_centr[df_centr["id_parcel"].isin(df_wide.index)]

print(f"Centroïdes : {len(df_centr):,} parcelles")
print(f"Emprise X : {df_centr['cx'].min():.0f} → {df_centr['cx'].max():.0f} m")
print(f"Emprise Y : {df_centr['cy'].min():.0f} → {df_centr['cy'].max():.0f} m")

In [ ]:
# --- Split spatial par blocs -----------------------------------------

BLOCK_SIZE = 10_000  # 10 km
TEST_RATIO = 0.20
SEED = 42

# Affecter chaque parcelle à un bloc (identifié par indices grille)
x_min, y_min = df_centr["cx"].min(), df_centr["cy"].min()
df_centr["block_x"] = ((df_centr["cx"] - x_min) // BLOCK_SIZE).astype(int)
df_centr["block_y"] = ((df_centr["cy"] - y_min) // BLOCK_SIZE).astype(int)
df_centr["block_id"] = df_centr["block_x"].astype(str) + "_" + df_centr["block_y"].astype(str)

# Tirage aléatoire des blocs test
blocks = df_centr["block_id"].unique()
rng = np.random.default_rng(SEED)
n_test_blocks = max(1, int(len(blocks) * TEST_RATIO))
test_blocks = set(rng.choice(blocks, size=n_test_blocks, replace=False))

df_centr["split"] = df_centr["block_id"].apply(
    lambda b: "test" if b in test_blocks else "train"
)

# Reporter le split sur df_wide (idempotent)
df_wide = df_wide.drop(columns=["split"], errors="ignore")
df_wide = df_wide.join(
    df_centr.set_index("id_parcel")[["split"]],
    on="id_parcel",
)

n_train = (df_wide["split"] == "train").sum()
n_test = (df_wide["split"] == "test").sum()
print(f"Blocs : {len(blocks)} total, {n_test_blocks} test")
print(f"Train : {n_train:,} ({100*n_train/len(df_wide):.1f} %)")
print(f"Test  : {n_test:,}  ({100*n_test/len(df_wide):.1f} %)")

In [ ]:
# --- Représentation par classe dans train / test ---------------------

ct = pd.crosstab(df_wide["classe"], df_wide["split"])
ct["total"] = ct.sum(axis=1)
ct["pct_test"] = (100 * ct["test"] / ct["total"]).round(1)
ct = ct.sort_values("total", ascending=False)

print(ct.to_string())

### 4.3 — Entraînement Random Forest

Baseline de classification : Random Forest scikit-learn sur le
feature set de 704 colonnes.

**Préparation** : les matrices `X_train`, `X_test` (features) et
`y_train`, `y_test` (classe cible) sont construites à partir du
split spatial défini en 4.2. Aucune normalisation n'est nécessaire
— Random Forest est invariant aux échelles des features.

**Hyperparamètres** : un premier entraînement utilise des
hyperparamètres par défaut, suivi d'une recherche par
`RandomizedSearchCV` (validation croisée sur le train uniquement)
pour ajuster `n_estimators`, `max_depth`, `min_samples_leaf` et
`max_features`. Le jeu test reste intouché jusqu'à l'évaluation
finale.

**Feature importances** : le modèle entraîné fournit un classement
des features les plus discriminantes, permettant d'identifier les
variables spectrales et les mois les plus utiles à la séparation
des classes.

In [ ]:
# --- Matrices X / y -------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

feature_cols = [c for c in df_wide.columns if c not in ("classe", "split")]

train_mask = df_wide["split"] == "train"
test_mask = df_wide["split"] == "test"

X_train = df_wide.loc[train_mask, feature_cols].values
X_test = df_wide.loc[test_mask, feature_cols].values
y_train = df_wide.loc[train_mask, "classe"].to_numpy()
y_test = df_wide.loc[test_mask, "classe"].to_numpy()

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {np.unique(y_train).tolist()}")

In [ ]:
# --- Random Forest baseline ------------------------------------------

rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf_base.fit(X_train, y_train)

score_train = rf_base.score(X_train, y_train)
score_test = rf_base.score(X_test, y_test)

print(f"Accuracy train : {score_train:.4f}")
print(f"Accuracy test  : {score_test:.4f}")

In [ ]:
# --- Matrice de confusion baseline -----------------------------------

from sklearn.metrics import confusion_matrix, classification_report

y_pred_base = rf_base.predict(X_test)

classes = sorted(np.unique(y_train))
cm = confusion_matrix(y_test, y_pred_base, labels=classes)

# Affichage formaté
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print("Matrice de confusion (lignes = réel, colonnes = prédit) :\n")
print(cm_df.to_string())
print(f"\n{classification_report(y_test, y_pred_base, digits=3)}")

In [ ]:
# --- RandomizedSearchCV ----------------------------------------------

param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [20, 30, 40],
    "min_samples_leaf": [2, 5, 10],
    "max_features": ["sqrt", 0.1, 0.2],
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=3,
    random_state=SEED,
    n_jobs=1,       # parallélisme au niveau du RF, pas du CV
    verbose=2,      # affiche le timing de chaque fit
)
search.fit(X_train, y_train)

print(f"\nMeilleur F1 macro (CV) : {search.best_score_:.4f}")
print(f"Meilleurs paramètres  : {search.best_params_}")

In [ ]:
# --- Évaluation modèle tuné -----------------------------------------

rf = search.best_estimator_

y_pred = rf.predict(X_test)

classes = sorted(np.unique(y_train))
cm = confusion_matrix(y_test, y_pred, labels=classes)
cm_df = pd.DataFrame(cm, index=classes, columns=classes)

print("Matrice de confusion (lignes = réel, colonnes = prédit) :\n")
print(cm_df.to_string())
print(f"\n{classification_report(y_test, y_pred, digits=3)}")

# Top 20 features
feature_cols = [c for c in df_wide.columns if c not in ("classe", "split")]
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)
print("\nTop 20 features :\n")
for feat, imp in top20.items():
    print(f"  {feat:<30s} {imp:.4f}")